# Gold Future and Option Open Interest Analysis

## Introduction

This Colab Notebook is designed to fetch, parse, validate, and display open interest data for gold futures and options from specified API endpoints.

**Purpose:**
The primary goal is to provide a structured way to retrieve financial market data and transform it into a usable format (pandas DataFrames) for further analysis, visualization, or reporting. It emphasizes a step-by-step process, including robust error handling during data fetching and clear validation checks.

**How to Use This Notebook:**
1.  **Configuration (Crucial):** Start by editing the code cell in "Section 1: Configuration and Data Fetching". You **must** replace the placeholder values for `API_KEY`, `GOLD_FUTURES_API_ENDPOINT`, and `GOLD_OPTIONS_API_ENDPOINT` with your actual API key and the correct URLs for the data sources you intend to use.
2.  **Run Cells Sequentially:** Execute the cells one by one from top to bottom. Each cell builds upon the previous ones (e.g., data fetching is needed before parsing, parsing before validation).
3.  **Adapt Parsing Functions (Very Important):** The functions in "Section 2: Parsing API Responses" (`parse_futures_data` and `parse_options_data`) are **templates**. API responses vary greatly. You will likely need to modify the logic within these functions to correctly extract data based on the specific JSON structure provided by your chosen API.
4.  **Review Validation Output:** Pay attention to messages in "Section 3: Data Validation". Warnings may indicate correctable issues (like data type conversions), while errors might point to more significant problems in the data or parsing logic.
5.  **View Displayed Data:** "Section 4: Displaying Fetched and Validated Data" will show the final DataFrames. If these are empty or incorrect, revisit the previous sections, especially parsing and configuration.
6.  **Iterate and Customize:** This notebook provides a foundation. Feel free to customize it for different financial instruments, add more sophisticated analysis, or integrate data visualization libraries.

## 1. Configuration and Data Fetching

This section initializes necessary libraries and sets up the configuration for API access. It includes:
- Importing Python libraries (`requests` for HTTP calls, `json` for handling JSON data, `pandas` for data manipulation).
- **User Configuration:** Placeholders for your API key and the specific API endpoints for gold futures and options data. **You must replace these with your actual values.**
- A helper function `fetch_data` to retrieve data from the specified API URLs. This function includes error handling for common HTTP issues.
- Example usage of the `fetch_data` function (commented out initially). Once you've configured your API key and endpoints, you can uncomment these lines to test data fetching.

In [ ]:
import requests
import json
import pandas as pd

# --- Configuration ---
# IMPORTANT: Replace the placeholder strings below with your actual API key and endpoints.
API_KEY = "YOUR_API_KEY_HERE"  # @param {type:"string"} # Your API authentication key.
GOLD_FUTURES_API_ENDPOINT = "YOUR_FUTURES_API_ENDPOINT_HERE"  # @param {type:"string"} # Full URL for futures data.
GOLD_OPTIONS_API_ENDPOINT = "YOUR_OPTIONS_API_ENDPOINT_HERE"  # @param {type:"string"} # Full URL for options data.

# --- Helper Function to Fetch Data ---
def fetch_data(api_url, api_key=None, params=None):
    """
    Fetches data from the specified API endpoint using an HTTP GET request.

    Args:
        api_url (str): The URL of the API endpoint to query.
        api_key (str, optional): The API key for authentication. If provided, it's typically sent
                                 in the request headers (e.g., as a Bearer token or via a custom header like 'X-Api-Key').
                                 Defaults to None if the API does not require a key or it's included directly in the URL.
        params (dict, optional): Dictionary of parameters to send in the query string of the request.
                                 Defaults to None.

    Returns:
        dict: The JSON response from the API parsed into a Python dictionary, 
              or None if an error occurs (e.g., network issue, bad HTTP status, JSON decoding error).
    """
    headers = {}
    if api_key:
        # Common ways to pass API keys:
        # 1. Bearer Token: 'Authorization': 'Bearer YOUR_API_KEY'
        # 2. Custom Header: 'X-Api-Key': 'YOUR_API_KEY' or 'apikey': 'YOUR_API_KEY'
        # Adjust the line below if your API expects a different header format for the API key.
        headers['Authorization'] = f'Bearer {api_key}' 
        # Example for a custom header: headers['X-Api-Key'] = api_key

    print(f"Fetching data from: {api_url}")
    if params:
        print(f"With parameters: {params}")

    try:
        # Make the GET request with a timeout (in seconds)
        response = requests.get(api_url, headers=headers, params=params, timeout=30)
        response.raise_for_status()  # Raises an HTTPError for bad responses (4XX client error or 5XX server error)
        print("Data fetched successfully.")
        return response.json()  # Parse the JSON response body into a Python dict
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err} - Status Code: {response.status_code}")
        print(f"Response content: {response.content}") # Show error response from API if any
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Connection error occurred: {conn_err} (Is the API URL correct and the server reachable?)")
    except requests.exceptions.Timeout as timeout_err:
        print(f"Timeout error occurred: {timeout_err} (The server did not respond within 30 seconds.)")
    except requests.exceptions.RequestException as req_err:
        print(f"An unexpected error occurred during the API request: {req_err}")
    except json.JSONDecodeError as json_err:
        print(f"JSON decoding error: {json_err}. The API response was not valid JSON.")
        print(f"Response text (first 500 chars): {response.text[:500]}")

    return None # Return None if any exception occurred

# --- Example Usage (Commented out by default) ---
# After configuring API_KEY and endpoints, uncomment these lines to test fetching data.
# print("\n--- Fetching Gold Futures Data (Test) ---")
# # Example parameters: Some APIs might require parameters like the specific contract symbol (e.g., 'GCZ3') or date.
# futures_params = {'symbol': 'GC', 'month': 'DEC23'} # Adjust parameters as needed for your API
# futures_data = fetch_data(GOLD_FUTURES_API_ENDPOINT, API_KEY, params=futures_params)
# if futures_data:
#     print("Futures Data Received (first 500 chars):", str(futures_data)[:500])

# print("\n--- Fetching Gold Options Data (Test) ---")
# # Example parameters for options often include underlying symbol, expiry, strike, type (call/put).
# options_params = {'underlyingSymbol': 'GC', 'expiry': '2023-12-26'} # Adjust parameters as needed
# options_data = fetch_data(GOLD_OPTIONS_API_ENDPOINT, API_KEY, params=options_params)
# if options_data:
#     print("Options Data Received (first 500 chars):", str(options_data)[:500])


## 2. Parsing API Responses

These functions parse the raw JSON data (obtained from `fetch_data`) into structured pandas DataFrames, which are easier to work with for analysis.

**VERY IMPORTANT:** 
The parsing logic within `parse_futures_data` and `parse_options_data` is **highly dependent on the specific JSON structure returned by YOUR API**. The provided code contains placeholder logic and attempts to guess common JSON patterns. 

**You MUST inspect the actual JSON response from your API and adapt these functions accordingly.**
1.  Run the `fetch_data` examples in the previous section (once configured) to see the raw JSON output.
2.  Identify the keys and nesting in the JSON that correspond to the data you need (e.g., contract symbol, open interest, expiry date, strike price, option type).
3.  Modify the lines commented with `TODO: Adapt the parsing logic...` and the subsequent data extraction loop.

The example usage below demonstrates these functions using **dummy data**. This allows you to see the expected DataFrame structure even before you've successfully fetched live data.

In [ ]:
# --- Functions to Parse API Responses ---

def parse_futures_data(api_response):
    """
    Parses the API response for gold futures data into a pandas DataFrame.
    This function is a TEMPLATE and almost certainly needs to be adapted 
    based on the actual structure of the API response you receive.

    Args:
        api_response (dict): The JSON response from the API, typically the output of `fetch_data()`.

    Returns:
        pandas.DataFrame: A DataFrame containing structured futures data with columns like 
                          'Symbol', 'Open Interest', 'Expiry Date'. 
                          Returns an empty DataFrame if parsing fails or no data is found.
    """
    if not api_response:
        print("No API response to parse for futures. Returning empty DataFrame.")
        return pd.DataFrame()

    try:
        # TODO: CRITICAL - Adapt the parsing logic below based on YOUR API's JSON structure.
        # Inspect `api_response` to find where the list of contracts and their details are located.
        # Example: if data is in response['resultsList'], use api_response.get('resultsList', [])
        
        # This checks if we're using the dummy data defined later in this cell for demonstration.
        if 'example_futures_data' in api_response: 
            futures_list = api_response['example_futures_data']
        else:
            # --- START OF SPECULATIVE PARSING --- 
            # This is a guess. Your API might structure data differently.
            if isinstance(api_response, list): # Case 1: The entire response is a list of contracts
                 futures_list = api_response
            elif 'data' in api_response and isinstance(api_response['data'], list): # Case 2: { "data": [...] }
                 futures_list = api_response['data']
            elif 'results' in api_response and isinstance(api_response['results'], list): # Case 3: { "results": [...] }
                 futures_list = api_response['results']
            # Add more elif blocks here for other potential structures, e.g.:
            # elif 'contracts' in api_response and isinstance(api_response['contracts'], list):
            #     futures_list = api_response['contracts']
            else:
                print("Could not identify a list of futures contracts in the API response. Please adapt parsing logic in parse_futures_data.")
                print("Response snippet (first 500 chars):", str(api_response)[:500])
                return pd.DataFrame() # Return empty if structure is unknown
            # --- END OF SPECULATIVE PARSING ---

        if not futures_list: # Check if the extracted list is empty
            print("No futures contracts found in the response after initial parsing.")
            return pd.DataFrame()

        parsed_data = []
        for contract in futures_list:
            if isinstance(contract, dict):
                # TODO: Adapt these .get() calls to match the field names in YOUR API's contract objects.
                # Common names for symbol: 'symbol', 'contractSymbol', 'ticker'
                symbol = contract.get('symbol', 'N/A') 
                # Common names for open interest: 'openInterest', 'oi', 'open_interest'
                open_interest = contract.get('open_interest', contract.get('oi', None))
                # Common names for expiry date: 'expiryDate', 'expirationDate', 'maturityDate', 'expDate'
                expiry_date = contract.get('expiry_date', contract.get('expirationDate', None))
                
                if open_interest is not None: # Only add contracts if essential data like open interest is present
                    parsed_data.append({
                        'Symbol': symbol,
                        'Open Interest': open_interest,
                        'Expiry Date': expiry_date
                    })
                else:
                    print(f"Skipping futures contract due to missing 'open_interest' or equivalent: {contract}")
            else:
                print(f"Skipping non-dictionary item in futures_list: {contract}")

        if not parsed_data:
            print("No futures contracts could be successfully parsed from the response.")
            return pd.DataFrame()
            
        df = pd.DataFrame(parsed_data)
        print(f"Successfully parsed {len(df)} futures contracts into a DataFrame.")
        return df

    except Exception as e:
        print(f"Error parsing futures data: {e}. This often occurs if the API response structure is not as expected.")
        print("Response snippet (first 500 chars):", str(api_response)[:500])
        return pd.DataFrame() # Return empty DataFrame on error

def parse_options_data(api_response):
    """
    Parses the API response for gold options data into a pandas DataFrame.
    This function is a TEMPLATE and almost certainly needs to be adapted 
    based on the actual structure of the API response you receive.

    Args:
        api_response (dict): The JSON response from the API, typically the output of `fetch_data()`.

    Returns:
        pandas.DataFrame: A DataFrame containing structured options data with columns like 'Symbol', 
                          'Strike Price', 'Type' (Call/Put), 'Open Interest', 'Expiry Date'.
                          Returns an empty DataFrame if parsing fails or no data is found.
    """
    if not api_response:
        print("No API response to parse for options. Returning empty DataFrame.")
        return pd.DataFrame()

    try:
        # TODO: CRITICAL - Adapt the parsing logic below based on YOUR API's JSON structure.
        if 'example_options_data' in api_response: # Using dummy data
            options_list = api_response['example_options_data']
        else:
            # --- START OF SPECULATIVE PARSING ---
            if isinstance(api_response, list):
                options_list = api_response
            elif 'data' in api_response and isinstance(api_response['data'], list):
                options_list = api_response['data']
            elif 'results' in api_response and isinstance(api_response['results'], list):
                options_list = api_response['results']
            # Add more elif blocks here for other potential structures
            else:
                print("Could not identify a list of options contracts in the API response. Please adapt parsing logic in parse_options_data.")
                print("Response snippet (first 500 chars):", str(api_response)[:500])
                return pd.DataFrame()
            # --- END OF SPECULATIVE PARSING ---

        if not options_list:
            print("No options contracts found in the response after initial parsing.")
            return pd.DataFrame()

        parsed_data = []
        for contract in options_list:
            if isinstance(contract, dict):
                # TODO: Adapt these .get() calls to match field names in YOUR API's option contract objects.
                symbol = contract.get('symbol', 'N/A')
                open_interest = contract.get('open_interest', contract.get('oi', None))
                strike_price = contract.get('strike_price', contract.get('strike', None))
                # Common names for option type: 'optionType', 'type', 'putCall' (ensure values are 'Call'/'Put' or 'C'/'P')
                option_type = contract.get('option_type', contract.get('type', None))
                expiry_date = contract.get('expiry_date', contract.get('expirationDate', None))

                # Essential fields for options data
                if open_interest is not None and strike_price is not None and option_type is not None:
                    parsed_data.append({
                        'Symbol': symbol,
                        'Strike Price': strike_price,
                        'Type': option_type.capitalize() if isinstance(option_type, str) else option_type, # Standardize to 'Call' or 'Put'
                        'Open Interest': open_interest,
                        'Expiry Date': expiry_date
                    })
                else:
                    print(f"Skipping options contract due to missing essential fields (open_interest, strike_price, option_type): {contract}")
            else:
                 print(f"Skipping non-dictionary item in options_list: {contract}")

        if not parsed_data:
            print("No option contracts could be successfully parsed from the response.")
            return pd.DataFrame()

        df = pd.DataFrame(parsed_data)
        print(f"Successfully parsed {len(df)} options contracts into a DataFrame.")
        return df

    except Exception as e:
        print(f"Error parsing options data: {e}. This often occurs if the API response structure is not as expected.")
        print("Response snippet (first 500 chars):", str(api_response)[:500])
        return pd.DataFrame()

# --- Example Usage with Dummy Data ---
# This section uses predefined dummy data to demonstrate the parsing functions.
# This is helpful for understanding the expected DataFrame structure and for testing parsing logic
# before you have live API data.

print("\n--- Parsing Example Gold Futures Data (using DUMMY data) ---")
# This is NOT a live API call. It uses the 'dummy_futures_response' dictionary below.
dummy_futures_response = {
    "example_futures_data": [
        {"symbol": "GCZ3", "open_interest": 150000, "expiry_date": "2023-12-28"},
        {"symbol": "GCG4", "open_interest": 120000, "expiry_date": "2024-02-27"},
        {"symbol": "GCM4", "oi": "N/A"}, # Example of open interest that might not be parseable as a number initially
        {"symbol": "GCV4", "open_interest": 90000} # Missing expiry_date (will be None/NaT)
    ]
}
futures_df = parse_futures_data(dummy_futures_response) # Call parsing function with dummy data
if not futures_df.empty:
    print("Parsed Dummy Futures Data (first 5 rows):")
    print(futures_df.head())

print("\n--- Parsing Example Gold Options Data (using DUMMY data) ---")
# This is NOT a live API call. It uses the 'dummy_options_response' dictionary below.
dummy_options_response = {
    "example_options_data": [
        {"symbol": "OGZ3C1800", "strike_price": 1800, "option_type": "Call", "open_interest": 5000, "expiry_date": "2023-12-26"},
        {"symbol": "OGZ3P1700", "strike_price": 1700, "type": "put", "oi": 3500, "expirationDate": "2023-12-26"}, # 'type' and 'oi' as alternative keys
        {"symbol": "OGG4C1850", "strike": 1850, "option_type": "CALL", "open_interest": 0}, # OI is zero, type is uppercase
        {"symbol": "OGG4P1750", "strike_price": 1750, "option_type": "Put"} # Missing open_interest (will be skipped by parser)
    ]
}
options_df = parse_options_data(dummy_options_response) # Call parsing function with dummy data
if not options_df.empty:
    print("Parsed Dummy Options Data (first 5 rows):")
    print(options_df.head())


## 3. Data Validation

These functions perform basic validation checks on the DataFrames created by the parsing functions.
The goals of validation are to:
- Ensure required columns are present.
- Check if data types are appropriate (e.g., 'Open Interest' should be numeric).
- Identify potential logical inconsistencies (e.g., negative open interest, non-positive strike prices).

**Interpreting Validation Messages:**
- **Validation Error:** Indicates a significant issue that likely needs to be addressed by correcting the parsing logic or investigating the source data (e.g., missing critical columns, negative open interest).
- **Validation Warning:** Suggests a potential issue or an automatic correction that was made (e.g., converting a column to a numeric type, missing non-critical data like 'Expiry Date'). These may or may not require action, depending on your analysis needs.

The functions will attempt to convert some columns to their expected types (e.g., using `pd.to_numeric`). Rows that cannot be converted correctly might result in `NaN` (Not a Number) values in those columns, which will be flagged by warnings.

In [ ]:
# --- Data Validation Functions ---

def validate_futures_data(df):
    """
    Validates the parsed gold futures DataFrame for required columns, data types,
    and logical consistency.

    Args:
        df (pandas.DataFrame): DataFrame containing futures data, typically from `parse_futures_data()`.

    Returns:
        bool: True if basic validation passes (or only warnings are issued), 
              False if critical validation errors are found.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("Validation: Futures DataFrame is not a valid DataFrame or is empty. Nothing to validate.")
        return True # Consider empty DataFrame as trivially valid or False if it's an error condition

    is_critically_valid = True # Tracks if any major errors occur
    print("\n--- Validating Futures Data ---")

    # Check for required columns
    required_columns = ['Symbol', 'Open Interest'] # 'Expiry Date' is useful but might be optional for some analyses
    for col in required_columns:
        if col not in df.columns:
            print(f"Validation Error: Missing required column: '{col}'. This may indicate a parsing issue.")
            is_critically_valid = False
    if not is_critically_valid: # Stop further checks if critical columns are missing
        print("Halting futures validation due to missing critical columns.")
        return False

    # Validate 'Symbol' column
    if not pd.api.types.is_string_dtype(df['Symbol']):
        try:
            df['Symbol'] = df['Symbol'].astype(str)
            print("Validation Warning: 'Symbol' column was not string type, converted to string.")
        except Exception as e:
            print(f"Validation Error: 'Symbol' column is not string and could not be converted: {e}")
            is_critically_valid = False
            
    # Validate 'Open Interest'
    # Attempt to convert to numeric, coercing errors to NaN (Not a Number)
    # This handles cases where 'Open Interest' might be a string like "N/A" or empty.
    df['Open Interest'] = pd.to_numeric(df['Open Interest'], errors='coerce')
    if df['Open Interest'].isnull().any():
        print(f"Validation Warning: Some 'Open Interest' values are not numeric (or were missing) and have been set to NaN. Review these rows:")
        print(df[df['Open Interest'].isnull()])
        # Depending on requirements, you might set is_critically_valid = False here
        
    if (df['Open Interest'] < 0).any():
        print(f"Validation Error: Negative 'Open Interest' values found, which is generally invalid. Review these rows:")
        print(df[df['Open Interest'] < 0])
        is_critically_valid = False

    # Validate 'Expiry Date' (optional column, so check existence first)
    if 'Expiry Date' in df.columns:
        # Attempt to convert to datetime objects. `errors='coerce'` will turn unparseable dates into NaT (Not a Time).
        original_expiry_null_count = df['Expiry Date'].isnull().sum()
        df['Expiry Date'] = pd.to_datetime(df['Expiry Date'], errors='coerce')
        new_expiry_null_count = df['Expiry Date'].isnull().sum()
        if new_expiry_null_count > original_expiry_null_count:
             print("Validation Warning: Some 'Expiry Date' values were not in a recognizable date format and were set to NaT (Not a Time).")
        if df['Expiry Date'].isnull().any():
            print("Validation Info: Some 'Expiry Date' values are missing (NaN/NaT) or were unparseable.")

    if is_critically_valid:
        print("Futures data validation completed. Check warnings above if any.")
    else:
        print("Futures data validation failed due to critical errors.")
    return is_critically_valid

def validate_options_data(df):
    """
    Validates the parsed gold options DataFrame for required columns, data types,
    and logical consistency.

    Args:
        df (pandas.DataFrame): DataFrame containing options data, from `parse_options_data()`.

    Returns:
        bool: True if basic validation passes, False if critical errors are found.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("Validation: Options DataFrame is not a valid DataFrame or is empty. Nothing to validate.")
        return True

    is_critically_valid = True
    print("\n--- Validating Options Data ---")

    required_columns = ['Symbol', 'Strike Price', 'Type', 'Open Interest'] # 'Expiry Date' is useful but optional
    for col in required_columns:
        if col not in df.columns:
            print(f"Validation Error: Missing required column: '{col}'. This may indicate a parsing issue.")
            is_critically_valid = False
    if not is_critically_valid:
        print("Halting options validation due to missing critical columns.")
        return False

    # Validate 'Symbol'
    if not pd.api.types.is_string_dtype(df['Symbol']):
        try:
            df['Symbol'] = df['Symbol'].astype(str)
            print("Validation Warning: 'Symbol' column was not string type, converted.")
        except Exception as e:
            print(f"Validation Error: 'Symbol' column is not string and could not be converted: {e}")
            is_critically_valid = False
            
    # Validate 'Strike Price'
    df['Strike Price'] = pd.to_numeric(df['Strike Price'], errors='coerce')
    if df['Strike Price'].isnull().any():
        print(f"Validation Warning: Some 'Strike Price' values are not numeric (or were missing) and were set to NaN. Review these rows:")
        print(df[df['Strike Price'].isnull()])
    if (df['Strike Price'] <= 0).any(): # Strike prices should be positive
        print(f"Validation Error: Non-positive 'Strike Price' values found. Review these rows:")
        print(df[df['Strike Price'] <=0])
        is_critically_valid = False
        
    # Validate 'Type' (Option Type)
    if 'Type' in df.columns:
        df['Type'] = df['Type'].astype(str).str.strip().str.capitalize() # Standardize: 'call' -> 'Call', 'PUT' -> 'Put'
        valid_types = ['Call', 'Put', 'C', 'P'] # Allow for common short forms as well
        # Create a boolean Series for rows with invalid types, excluding NaN values which are handled by 'missing'
        invalid_type_mask = ~df['Type'].isin(valid_types) & df['Type'].notna()
        if invalid_type_mask.any():
            print(f"Validation Warning: 'Type' column contains values not in {valid_types}. Review these rows:")
            print(df[invalid_type_mask])
            # is_critically_valid = False # Decide if this is critical

    # Validate 'Open Interest'
    df['Open Interest'] = pd.to_numeric(df['Open Interest'], errors='coerce')
    if df['Open Interest'].isnull().any():
        print(f"Validation Warning: Some 'Open Interest' values are not numeric (or were missing) and were set to NaN. Review these rows:")
        print(df[df['Open Interest'].isnull()])
    if (df['Open Interest'] < 0).any():
        print(f"Validation Error: Negative 'Open Interest' values found. Review these rows:")
        print(df[df['Open Interest'] < 0])
        is_critically_valid = False

    # Validate 'Expiry Date' (optional)
    if 'Expiry Date' in df.columns:
        original_expiry_null_count = df['Expiry Date'].isnull().sum()
        df['Expiry Date'] = pd.to_datetime(df['Expiry Date'], errors='coerce')
        new_expiry_null_count = df['Expiry Date'].isnull().sum()
        if new_expiry_null_count > original_expiry_null_count:
             print("Validation Warning: Some 'Expiry Date' values were not in a recognizable date format and were set to NaT.")
        if df['Expiry Date'].isnull().any():
            print("Validation Info: Some 'Expiry Date' values are missing (NaN/NaT) or were unparseable.")

    if is_critically_valid:
        print("Options data validation completed. Check warnings above if any.")
    else:
        print("Options data validation failed due to critical errors.")
    return is_critically_valid

# --- Example Usage with the Dummy DataFrames ---
# This demonstrates the validation functions using the 'futures_df' and 'options_df'
# created from dummy data in the previous section.

print("\n--- Validating Parsed Futures Data (from DUMMY data) ---")
if 'futures_df' in globals() and isinstance(futures_df, pd.DataFrame) and not futures_df.empty:
    futures_valid = validate_futures_data(futures_df.copy()) # Use .copy() to avoid modifying the original df in validation
    print(f"Futures data critically valid: {futures_valid}")
    print("Futures DataFrame after validation attempts (dtypes and first 5 rows):")
    print(futures_df.info()) 
    print(futures_df.head())
else:
    print("futures_df is not defined or is empty. Skipping validation example.")

print("\n--- Validating Parsed Options Data (from DUMMY data) ---")
if 'options_df' in globals() and isinstance(options_df, pd.DataFrame) and not options_df.empty:
    options_valid = validate_options_data(options_df.copy()) # Use .copy()
    print(f"Options data critically valid: {options_valid}")
    print("Options DataFrame after validation attempts (dtypes and first 5 rows):")
    print(options_df.info())
    print(options_df.head())
else:
    print("options_df is not defined or is empty. Skipping validation example.")


## 4. Displaying Fetched and Validated Data

This section displays the final, processed pandas DataFrames (`futures_df` and `options_df`). 
If the previous steps (fetching, parsing, validation) were successful, these DataFrames will contain the structured gold futures and options data.

The code below will:
- Print the entire content of `futures_df` if it exists and is not empty.
- Print the entire content of `options_df` if it exists and is not empty.
- If a DataFrame is missing or empty (e.g., due to fetching or parsing errors), it will print a message indicating this and, if available, show a snippet of the raw data that was fetched (if the `fetch_data` step succeeded but parsing failed).

For large DataFrames, `to_string()` ensures all rows are printed. In a Colab environment, you can also use `display(your_dataframe)` for a more interactive, paginated table (example commented out).

In [ ]:
# --- Displaying the DataFrames ---

# This cell assumes that 'futures_df' and 'options_df' have been populated
# by the parsing functions and potentially modified by validation functions.
# It also assumes 'futures_data' and 'options_data' might hold raw fetched data.

print("\n--- Displaying Final Gold Futures Data ---")
if 'futures_df' in globals() and isinstance(futures_df, pd.DataFrame) and not futures_df.empty:
    print("Processed and Validated Gold Futures Data:")
    # Ensure 'Open Interest' is numeric for consistent display, handling potential earlier issues if validation didn't run or modify df in place.
    futures_df_display = futures_df.copy() # Work on a copy for display modifications
    futures_df_display['Open Interest'] = pd.to_numeric(futures_df_display['Open Interest'], errors='coerce')
    if 'Expiry Date' in futures_df_display.columns:
        futures_df_display['Expiry Date'] = pd.to_datetime(futures_df_display['Expiry Date'], errors='coerce').dt.strftime('%Y-%m-%d') # Format date
    print(futures_df_display.to_string()) # .to_string() prints the entire DataFrame
else:
    print("No processed futures data to display (DataFrame is missing, empty, or not valid).")
    # Check if raw data was fetched but parsing might have failed
    if 'dummy_futures_response' in globals() and futures_df.empty: # Check dummy first if it was run
        print("Dummy futures_response was used, and parsing resulted in an empty DataFrame. Check parsing/validation logic.")
    elif 'futures_data' in globals() and futures_data is not None:
        print("Raw fetched futures_data was available but futures_df is empty. This suggests a parsing or validation issue.")
        print("Raw data snippet (first 1000 chars):", str(futures_data)[:1000])

print("\n--- Displaying Final Gold Options Data ---")
if 'options_df' in globals() and isinstance(options_df, pd.DataFrame) and not options_df.empty:
    print("Processed and Validated Gold Options Data:")
    options_df_display = options_df.copy()
    options_df_display['Open Interest'] = pd.to_numeric(options_df_display['Open Interest'], errors='coerce')
    options_df_display['Strike Price'] = pd.to_numeric(options_df_display['Strike Price'], errors='coerce')
    if 'Expiry Date' in options_df_display.columns:
        options_df_display['Expiry Date'] = pd.to_datetime(options_df_display['Expiry Date'], errors='coerce').dt.strftime('%Y-%m-%d')
    print(options_df_display.to_string()) 
else:
    print("No processed options data to display (DataFrame is missing, empty, or not valid).")
    if 'dummy_options_response' in globals() and options_df.empty:
        print("Dummy options_response was used, and parsing resulted in an empty DataFrame. Check parsing/validation logic.")
    elif 'options_data' in globals() and options_data is not None:
        print("Raw fetched options_data was available but options_df is empty. This suggests a parsing or validation issue.")
        print("Raw data snippet (first 1000 chars):", str(options_data)[:1000])

# --- Colab Specific Interactive Display (Optional) ---
# You can uncomment the lines below to use Google Colab's interactive data table display.
# from google.colab.data_table import DataTable
# if 'futures_df_display' in globals() and not futures_df_display.empty:
#    print("\n--- Interactive Futures Table (Colab Specific) ---")
#    display(DataTable(futures_df_display)) # display() is a Colab built-in function
# if 'options_df_display' in globals() and not options_df_display.empty:
#    print("\n--- Interactive Options Table (Colab Specific) ---")
#    display(DataTable(options_df_display))


## 5. Summary and Next Steps

This notebook has guided you through the process of:
1.  **Configuring API access:** Setting up API keys and endpoints.
2.  **Fetching data:** Retrieving raw data for gold futures and options using the `fetch_data` function.
3.  **Parsing data:** Transforming the raw JSON responses into structured pandas DataFrames (`futures_df`, `options_df`) using customizable parsing functions (`parse_futures_data`, `parse_options_data`). This step used dummy data for initial demonstration.
4.  **Validating data:** Checking the DataFrames for completeness, correct data types, and logical consistency using `validate_futures_data` and `validate_options_data`.
5.  **Displaying data:** Showing the final processed DataFrames.

**Next Steps & Potential Enhancements:**

-   **Use Live Data:** The most crucial next step is to replace the dummy API responses with actual calls to `fetch_data` using your configured API key and endpoints. This involves uncommenting and adapting the example usage in Section 1 and ensuring the parsing functions in Section 2 are correctly tailored to your API's response format.
-   **Advanced Analysis:**
    *   Calculate total open interest across different expiries or strike prices.
    *   Identify contracts with the highest open interest.
    *   Analyze the put/call ratio for options.
    *   Compare open interest changes over time (if you fetch data periodically).
-   **Data Visualization:** Use libraries like Matplotlib, Seaborn, or Plotly to create charts:
    *   Bar charts of open interest per expiry month (futures).
    *   Bar charts of open interest per strike price for calls and puts (options).
    *   Heatmaps of open interest by strike and expiry.
-   **Error Handling & Robustness:**
    *   Implement more sophisticated retry mechanisms in `fetch_data`.
    *   Add more detailed logging throughout the process.
-   **Saving Data:** Save the processed DataFrames to files (e.g., CSV, Excel) for offline use or for import into other tools:
    ```python
    # if 'futures_df' in globals() and not futures_df.empty:
    #     futures_df.to_csv('gold_futures_open_interest.csv', index=False)
    # if 'options_df' in globals() and not options_df.empty:
    #     options_df.to_csv('gold_options_open_interest.csv', index=False)
    ```
-   **Parameterization:** If using this notebook regularly, consider using Colab forms to input parameters like dates or contract symbols more easily, rather than editing code cells directly.

Remember to adapt the parsing and validation logic based on the specific characteristics of your data source.